# 01. Temporal Modeling: Medformer + Markov Chain ICU Trajectory Analysis

This notebook is designed for **Google Colab** and should be run from the Medformer project folder.

It does **analysis generation only**:

- Load ICU time-series CSV files from `data/`
- Build 48-hour / 4-hour-bin Medformer tensors
- Train and evaluate Medformer
- Build Markov-chain ICU trajectory states
- Extract Markov trajectory features
- Create one integrated patient-level summary table for figure generation

The publication figures are generated in the second notebook:

`02_temporal_figures_170mm_colab.ipynb`

In [ ]:
# ============================================================
# 0. Colab setup, Drive mount, and project paths
# ============================================================

import os
import sys
import glob
import time
import math
import copy
import random
import warnings
import subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

# ---- Google Colab Drive mount ----
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---- Change this only if your folder is somewhere else ----
DEFAULT_COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Documents/research/IUBDC 2026/Medformer")

if IN_COLAB and DEFAULT_COLAB_PROJECT_ROOT.exists():
    PROJECT_ROOT = DEFAULT_COLAB_PROJECT_ROOT
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "temporal_outputs"
DATA_OUT = OUTPUT_ROOT / "data"
RESULTS_DIR = OUTPUT_ROOT / "results"
FIG_DIR = OUTPUT_ROOT / "figures"

for p in [OUTPUT_ROOT, DATA_OUT, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# 1. Install minimal dependencies
# ============================================================

def pip_install_if_missing(package_name, import_name=None):
    import importlib
    import_name = import_name or package_name.split("==")[0].replace("-", "_")
    try:
        importlib.import_module(import_name)
        print(f"Already installed: {package_name}")
    except Exception:
        print(f"Installing: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Medformer import dependencies
pip_install_if_missing("reformer-pytorch==1.4.4", "reformer_pytorch")
pip_install_if_missing("einops==0.4.0", "einops")
pip_install_if_missing("natsort", "natsort")
pip_install_if_missing("tqdm", "tqdm")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# ============================================================
# 2. Resolve input CSV files
# ============================================================

def resolve_one(pattern):
    matches = sorted(glob.glob(str(DATA_DIR / pattern)))
    if not matches:
        raise FileNotFoundError(f"Could not find a file matching: {DATA_DIR / pattern}")
    return Path(matches[0])

train_ts_path = resolve_one("time_series_train_a_plus_b_train_c_test*.csv")
test_ts_path = resolve_one("time_series_test_a_plus_b_train_c_test*.csv")
y_train_path = resolve_one("time_series_y_train_a_plus_b_train_c_test*.csv")
y_test_path = resolve_one("time_series_y_test_a_plus_b_train_c_test*.csv")

print("Train time-series:", train_ts_path)
print("Test time-series :", test_ts_path)
print("Train labels     :", y_train_path)
print("Test labels      :", y_test_path)

In [ ]:
# ============================================================
# 3. Load raw CSV files and labels
# ============================================================

train_ts_df = pd.read_csv(train_ts_path)
test_ts_df = pd.read_csv(test_ts_path)
y_train_df = pd.read_csv(y_train_path)
y_test_df = pd.read_csv(y_test_path)

print("train_ts_df:", train_ts_df.shape)
print("test_ts_df :", test_ts_df.shape)
print("y_train_df :", y_train_df.shape)
print("y_test_df  :", y_test_df.shape)

def standardize_label_df(df):
    df = df.copy()
    lower_to_original = {c.lower(): c for c in df.columns}

    record_col = None
    for cand in ["recordid", "record_id", "id"]:
        if cand in lower_to_original:
            record_col = lower_to_original[cand]
            break
    if record_col is None:
        record_col = df.columns[0]

    label_col = None
    for cand in ["in_hospital_death", "label", "y", "outcome"]:
        if cand in lower_to_original:
            label_col = lower_to_original[cand]
            break
    if label_col is None:
        non_record_cols = [c for c in df.columns if c != record_col]
        label_col = non_record_cols[0]

    out = df[[record_col, label_col]].copy()
    out.columns = ["RecordID", "label"]
    out["RecordID"] = out["RecordID"].astype(int)
    out["label"] = out["label"].astype(int)
    return out

y_train_df = standardize_label_df(y_train_df)
y_test_df = standardize_label_df(y_test_df)

print("Train mortality prevalence:", y_train_df["label"].mean())
print("Test mortality prevalence :", y_test_df["label"].mean())

In [ ]:
# ============================================================
# 4. Shared temporal setup
# ============================================================

STATIC_VARS = ["Age", "Gender", "Height", "ICUType", "Weight"]

DYNAMIC_VARS = [
    "ALP", "ALT", "AST", "Albumin", "BUN", "Bilirubin", "Cholesterol", "Creatinine",
    "DiasABP", "FiO2", "GCS", "Glucose", "HCO3", "HCT", "HR", "K", "Lactate", "MAP",
    "MechVent", "Mg", "NIDiasABP", "NIMAP", "NISysABP", "Na", "PaCO2", "PaO2",
    "Platelets", "RespRate", "SaO2", "SysABP", "Temp", "TroponinI", "TroponinT",
    "Urine", "WBC", "pH"
]

SEQ_LEN = 12
BIN_MINUTES = 240
STATE_LABELS = ["Stable", "Moderate", "High", "Critical"]

print("Number of dynamic variables:", len(DYNAMIC_VARS))
print("Medformer input channels = values + masks:", len(DYNAMIC_VARS) * 2)

In [ ]:
# ============================================================
# 5. Standardize time-series dataframe and build value/mask tensors
# ============================================================

def standardize_ts_df(df):
    df = df.copy()
    lower_to_original = {c.lower(): c for c in df.columns}

    record_col = lower_to_original.get("recordid", None)
    parameter_col = lower_to_original.get("parameter", None)
    time_col = lower_to_original.get("time_minutes", None)

    # Prefer Value_numeric if present, otherwise use Value.
    # This avoids duplicated "Value_numeric" columns.
    if "value_numeric" in lower_to_original:
        value_col = lower_to_original["value_numeric"]
    elif "value" in lower_to_original:
        value_col = lower_to_original["value"]
    else:
        value_col = None

    missing = []
    if record_col is None:
        missing.append("RecordID")
    if parameter_col is None:
        missing.append("Parameter")
    if time_col is None:
        missing.append("time_minutes")
    if value_col is None:
        missing.append("Value_numeric or Value")

    if missing:
        raise ValueError(f"Missing required columns in time-series dataframe: {missing}")

    out = df[[record_col, parameter_col, time_col, value_col]].copy()
    out.columns = ["RecordID", "Parameter", "time_minutes", "Value_numeric"]

    out["RecordID"] = out["RecordID"].astype(int)
    out["Parameter"] = out["Parameter"].astype(str)
    out["time_minutes"] = pd.to_numeric(out["time_minutes"], errors="coerce")
    out["Value_numeric"] = pd.to_numeric(out["Value_numeric"], errors="coerce")
    out = out.dropna(subset=["RecordID", "Parameter", "time_minutes", "Value_numeric"])
    return out

train_ts_df = standardize_ts_df(train_ts_df)
test_ts_df = standardize_ts_df(test_ts_df)

print("Standardized train columns:", train_ts_df.columns.tolist())
print("Standardized test columns :", test_ts_df.columns.tolist())

def build_value_mask_tensors(ts_df, label_df, dynamic_vars, seq_len=12, bin_minutes=240):
    ts_df = ts_df[ts_df["Parameter"].isin(dynamic_vars)].copy()
    ts_df["bin_id"] = (ts_df["time_minutes"] // bin_minutes).astype(int)
    ts_df["bin_id"] = ts_df["bin_id"].clip(lower=0, upper=seq_len - 1)

    grouped = (
        ts_df.groupby(["RecordID", "bin_id", "Parameter"], as_index=False)["Value_numeric"]
        .mean()
    )

    record_ids = label_df["RecordID"].astype(int).tolist()
    id_to_idx = {rid: i for i, rid in enumerate(record_ids)}
    var_to_idx = {var: j for j, var in enumerate(dynamic_vars)}

    X_value = np.full((len(record_ids), seq_len, len(dynamic_vars)), np.nan, dtype=np.float32)
    X_mask = np.zeros((len(record_ids), seq_len, len(dynamic_vars)), dtype=np.float32)

    for row in grouped.itertuples(index=False):
        rid = int(row.RecordID)
        if rid not in id_to_idx:
            continue
        i = id_to_idx[rid]
        t = int(row.bin_id)
        j = var_to_idx[row.Parameter]
        X_value[i, t, j] = float(row.Value_numeric)
        X_mask[i, t, j] = 1.0

    y = label_df["label"].astype(int).to_numpy()
    return X_value, X_mask, np.array(record_ids, dtype=int), y

X_train_value_raw, X_train_mask_raw, record_ids_train_all, y_train_all = build_value_mask_tensors(
    train_ts_df, y_train_df, DYNAMIC_VARS, seq_len=SEQ_LEN, bin_minutes=BIN_MINUTES
)
X_test_value_raw, X_test_mask_raw, record_ids_test, y_test = build_value_mask_tensors(
    test_ts_df, y_test_df, DYNAMIC_VARS, seq_len=SEQ_LEN, bin_minutes=BIN_MINUTES
)

print("Raw train value tensor:", X_train_value_raw.shape)
print("Raw train mask tensor :", X_train_mask_raw.shape)
print("Raw test value tensor :", X_test_value_raw.shape)
print("Raw test mask tensor  :", X_test_mask_raw.shape)

In [ ]:
# ============================================================
# 6. Data QC outputs for supplementary figures
# ============================================================

def summarize_observation_counts(X_mask, record_ids, split_name):
    observed_entries = X_mask.sum(axis=(1, 2))
    possible_entries = X_mask.shape[1] * X_mask.shape[2]
    return pd.DataFrame({
        "split": split_name,
        "RecordID": record_ids,
        "observed_entries": observed_entries,
        "possible_entries": possible_entries,
        "observed_fraction": observed_entries / possible_entries,
    })

def summarize_temporal_coverage(X_mask, split_name):
    rows = []
    for b in range(X_mask.shape[1]):
        rows.append({
            "split": split_name,
            "bin_id": b,
            "hours": f"{4*b}-{4*(b+1)}h",
            "observed_fraction": float(X_mask[:, b, :].mean()),
        })
    return pd.DataFrame(rows)

def summarize_variable_time_coverage(X_mask, dynamic_vars, split_name):
    rows = []
    for b in range(X_mask.shape[1]):
        for j, var in enumerate(dynamic_vars):
            rows.append({
                "split": split_name,
                "bin_id": b,
                "hours": f"{4*b}-{4*(b+1)}h",
                "Parameter": var,
                "observed_fraction": float(X_mask[:, b, j].mean()),
            })
    return pd.DataFrame(rows)

def summarize_variable_coverage(X_mask, dynamic_vars, split_name):
    rows = []
    for j, var in enumerate(dynamic_vars):
        rows.append({
            "split": split_name,
            "Parameter": var,
            "observed_fraction": float(X_mask[:, :, j].mean()),
            "missing_fraction": float(1.0 - X_mask[:, :, j].mean()),
        })
    return pd.DataFrame(rows)

cohort_summary = pd.DataFrame([
    {"split": "train_AplusB_all", "n": len(y_train_all), "positive": int(y_train_all.sum()), "mortality_rate": float(y_train_all.mean())},
    {"split": "test_C", "n": len(y_test), "positive": int(y_test.sum()), "mortality_rate": float(y_test.mean())},
])
cohort_summary.to_csv(RESULTS_DIR / "cohort_summary.csv", index=False)

obs_counts = pd.concat([
    summarize_observation_counts(X_train_mask_raw, record_ids_train_all, "train_AplusB_all"),
    summarize_observation_counts(X_test_mask_raw, record_ids_test, "test_C"),
], ignore_index=True)
obs_counts.to_csv(RESULTS_DIR / "observation_count_by_patient.csv", index=False)

temporal_cov = pd.concat([
    summarize_temporal_coverage(X_train_mask_raw, "train_AplusB_all"),
    summarize_temporal_coverage(X_test_mask_raw, "test_C"),
], ignore_index=True)
temporal_cov.to_csv(RESULTS_DIR / "temporal_coverage_by_bin.csv", index=False)

var_time_cov = pd.concat([
    summarize_variable_time_coverage(X_train_mask_raw, DYNAMIC_VARS, "train_AplusB_all"),
    summarize_variable_time_coverage(X_test_mask_raw, DYNAMIC_VARS, "test_C"),
], ignore_index=True)
var_time_cov.to_csv(RESULTS_DIR / "variable_time_coverage.csv", index=False)

var_cov = pd.concat([
    summarize_variable_coverage(X_train_mask_raw, DYNAMIC_VARS, "train_AplusB_all"),
    summarize_variable_coverage(X_test_mask_raw, DYNAMIC_VARS, "test_C"),
], ignore_index=True)
var_cov.to_csv(RESULTS_DIR / "variable_coverage_summary.csv", index=False)

print("Saved QC outputs to:", RESULTS_DIR)

In [ ]:
# ============================================================
# 7. Train/validation split and leakage-free scaling
# ============================================================

idx_all = np.arange(len(y_train_all))
train_idx, val_idx = train_test_split(
    idx_all,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_all,
)

X_train_value_raw_split = X_train_value_raw[train_idx]
X_val_value_raw = X_train_value_raw[val_idx]
X_train_mask_split = X_train_mask_raw[train_idx]
X_val_mask = X_train_mask_raw[val_idx]

y_train = y_train_all[train_idx]
y_val = y_train_all[val_idx]
record_ids_train = record_ids_train_all[train_idx]
record_ids_val = record_ids_train_all[val_idx]

def fit_train_scaler(X_value_train):
    flat = X_value_train.reshape(-1, X_value_train.shape[-1])
    means = np.nanmean(flat, axis=0)
    stds = np.nanstd(flat, axis=0)
    stds = np.where(stds < 1e-8, 1.0, stds)
    return means.astype(np.float32), stds.astype(np.float32)

def apply_scaler_and_concat(X_value, X_mask, means, stds):
    X_scaled = (X_value - means[None, None, :]) / stds[None, None, :]
    X_scaled = np.where(np.isnan(X_scaled), 0.0, X_scaled).astype(np.float32)
    X_mask = X_mask.astype(np.float32)
    return np.concatenate([X_scaled, X_mask], axis=2).astype(np.float32)

means, stds = fit_train_scaler(X_train_value_raw_split)

X_train_seq = apply_scaler_and_concat(X_train_value_raw_split, X_train_mask_split, means, stds)
X_val_seq = apply_scaler_and_concat(X_val_value_raw, X_val_mask, means, stds)
X_test_seq = apply_scaler_and_concat(X_test_value_raw, X_test_mask_raw, means, stds)

np.save(DATA_OUT / "X_train_seq.npy", X_train_seq)
np.save(DATA_OUT / "X_val_seq.npy", X_val_seq)
np.save(DATA_OUT / "X_test_seq.npy", X_test_seq)
np.save(DATA_OUT / "y_train.npy", y_train)
np.save(DATA_OUT / "y_val.npy", y_val)
np.save(DATA_OUT / "y_test.npy", y_test)
pd.DataFrame({"RecordID": record_ids_train}).to_csv(DATA_OUT / "record_ids_train.csv", index=False)
pd.DataFrame({"RecordID": record_ids_val}).to_csv(DATA_OUT / "record_ids_val.csv", index=False)
pd.DataFrame({"RecordID": record_ids_test}).to_csv(DATA_OUT / "record_ids_test.csv", index=False)
pd.DataFrame({"feature_name": DYNAMIC_VARS}).to_csv(DATA_OUT / "feature_names.csv", index=False)

print("Train:", X_train_seq.shape, y_train.shape)
print("Val  :", X_val_seq.shape, y_val.shape)
print("Test :", X_test_seq.shape, y_test.shape)

In [ ]:
# ============================================================
# 8. PyTorch Dataset and DataLoader
# ============================================================

class ICUSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 128

train_loader = DataLoader(ICUSequenceDataset(X_train_seq, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(ICUSequenceDataset(X_val_seq, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(ICUSequenceDataset(X_test_seq, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
# ============================================================
# 9. Import Medformer model
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from models.Medformer import Model as MedformerModel
except ModuleNotFoundError as e:
    missing_pkg = str(e).replace("No module named ", "").replace("'", "")
    raise ModuleNotFoundError(
        f"Missing package or module: {missing_pkg}. "
        "Make sure this notebook is run from the Medformer repo root and dependencies are installed."
    ) from e

from types import SimpleNamespace

config = SimpleNamespace(
    task_name="classification",
    pred_len=0,
    output_attention=False,
    enc_in=X_train_seq.shape[2],
    seq_len=X_train_seq.shape[1],
    num_class=2,
    d_model=64,
    n_heads=4,
    e_layers=2,
    d_ff=128,
    dropout=0.1,
    activation="gelu",
    patch_len_list="2,4,6",
    augmentations="none",
    single_channel=False,
    no_inter_attn=False,
)

model = MedformerModel(config).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", f"{n_params:,}")
print("Input shape:", [BATCH_SIZE, config.seq_len, config.enc_in])

In [ ]:
# ============================================================
# 10. Medformer training utilities
# ============================================================

def evaluate_loader(model, loader):
    model.eval()
    all_probs, all_true, all_logits = [], [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            logits = model(batch_x, None, None, None)
            probs = torch.softmax(logits, dim=1)
            all_logits.append(logits.detach().cpu().numpy())
            all_probs.append(probs.detach().cpu().numpy())
            all_true.append(batch_y.numpy())

    probs = np.concatenate(all_probs, axis=0)
    logits = np.concatenate(all_logits, axis=0)
    y_true = np.concatenate(all_true, axis=0).astype(int)
    p_pos = probs[:, 1]
    y_pred = (p_pos >= 0.5).astype(int)

    metrics = {
        "AUROC": roc_auc_score(y_true, p_pos),
        "AUPRC": average_precision_score(y_true, p_pos),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, p_pos),
    }
    return metrics, y_true, p_pos, probs, logits

class BestStateTracker:
    def __init__(self, mode="max"):
        self.mode = mode
        self.best_score = None
        self.best_epoch = None
        self.best_state = None

    def update(self, score, model, epoch):
        if self.best_score is None:
            improved = True
        elif self.mode == "max":
            improved = score > self.best_score
        else:
            improved = score < self.best_score

        if improved:
            self.best_score = float(score)
            self.best_epoch = int(epoch)
            self.best_state = copy.deepcopy(model.state_dict())
        return improved

In [ ]:
# ============================================================
# 11. Train Medformer
# ============================================================

LEARNING_RATE = 1e-4
MAX_EPOCHS = 50
PATIENCE = 8

# Options: "none", "sqrt", "full"
CLASS_WEIGHT_MODE = "sqrt"

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
full_pos_weight = neg_count / max(pos_count, 1)

if CLASS_WEIGHT_MODE == "none":
    weight_values = [1.0, 1.0]
elif CLASS_WEIGHT_MODE == "sqrt":
    weight_values = [1.0, math.sqrt(full_pos_weight)]
elif CLASS_WEIGHT_MODE == "full":
    weight_values = [1.0, full_pos_weight]
else:
    raise ValueError("CLASS_WEIGHT_MODE must be one of: none, sqrt, full")

class_weights = torch.tensor(weight_values, dtype=torch.float32, device=DEVICE)
print("Class counts:", {"negative": neg_count, "positive": pos_count})
print("Class weights:", class_weights.detach().cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_auprc = BestStateTracker(mode="max")
best_brier = BestStateTracker(mode="min")
epochs_without_improvement = 0
history = []

start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    batch_losses = []

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(batch_x, None, None, None)
        loss = criterion(logits, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=4.0)
        optimizer.step()
        batch_losses.append(loss.item())

    train_loss = float(np.mean(batch_losses))
    val_metrics, _, _, _, _ = evaluate_loader(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_AUPRC": val_metrics["AUPRC"],
        "val_AUROC": val_metrics["AUROC"],
        "val_Accuracy": val_metrics["Accuracy"],
        "val_Precision": val_metrics["Precision"],
        "val_Recall": val_metrics["Recall"],
        "val_F1": val_metrics["F1"],
        "val_Brier": val_metrics["Brier"],
    }
    history.append(row)

    improved = best_auprc.update(val_metrics["AUPRC"], model, epoch)
    best_brier.update(val_metrics["Brier"], model, epoch)

    if improved:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d} | loss={train_loss:.4f} | "
        f"val_AUPRC={val_metrics['AUPRC']:.4f} | "
        f"val_AUROC={val_metrics['AUROC']:.4f} | "
        f"val_F1={val_metrics['F1']:.4f} | "
        f"val_Brier={val_metrics['Brier']:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

elapsed_min = (time.time() - start_time) / 60
print(f"Training finished in {elapsed_min:.2f} min")
print("Best AUPRC epoch:", best_auprc.best_epoch, "score:", best_auprc.best_score)
print("Best Brier epoch:", best_brier.best_epoch, "score:", best_brier.best_score)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "medformer_training_history.csv", index=False)

torch.save(best_auprc.best_state, RESULTS_DIR / "medformer_best_by_auprc.pt")
torch.save(best_brier.best_state, RESULTS_DIR / "medformer_best_by_brier.pt")

In [ ]:
# ============================================================
# 12. External test evaluation for selected Medformer checkpoints
# ============================================================

def evaluate_checkpoint(state_dict, model_selection_name):
    model.load_state_dict(state_dict)
    metrics, y_true, p_pos, probs, logits = evaluate_loader(model, test_loader)

    pred_df = pd.DataFrame({
        "RecordID": record_ids_test,
        "y_true": y_true,
        "p_mortality": p_pos,
        "y_pred": (p_pos >= 0.5).astype(int),
        "model_selection": model_selection_name,
        "split": "test_C",
    })
    metric_df = pd.DataFrame([{**metrics, "model_selection": model_selection_name}])
    return metric_df, pred_df, logits

metric_auprc, pred_auprc, logits_auprc = evaluate_checkpoint(best_auprc.best_state, "best_val_AUPRC")
metric_brier, pred_brier, logits_brier = evaluate_checkpoint(best_brier.best_state, "best_val_Brier")

medformer_metrics = pd.concat([metric_auprc, metric_brier], ignore_index=True)
medformer_preds_all = pd.concat([pred_auprc, pred_brier], ignore_index=True)

medformer_metrics.to_csv(RESULTS_DIR / "medformer_test_metrics.csv", index=False)
medformer_preds_all.to_csv(RESULTS_DIR / "medformer_test_predictions_all_checkpoints.csv", index=False)

# Main selected checkpoint for integrated figures
SELECTED_MODEL_SELECTION = "best_val_AUPRC"
medformer_pred_selected = medformer_preds_all[
    medformer_preds_all["model_selection"] == SELECTED_MODEL_SELECTION
].copy()
medformer_pred_selected.to_csv(RESULTS_DIR / "medformer_test_predictions.csv", index=False)

print(medformer_metrics)

In [ ]:
# ============================================================
# 13. Medformer temporal permutation importance
# ============================================================

model.load_state_dict(best_auprc.best_state)

def evaluate_array(model, X_array, y_array, batch_size=256):
    loader = DataLoader(ICUSequenceDataset(X_array, y_array), batch_size=batch_size, shuffle=False)
    metrics, _, _, _, _ = evaluate_loader(model, loader)
    return metrics

baseline_metrics = evaluate_array(model, X_test_seq, y_test, batch_size=256)
print("Baseline metrics:", baseline_metrics)

timebin_rows = []
for b in range(SEQ_LEN):
    X_perm = X_test_seq.copy()
    X_perm[:, b, :] = 0.0
    m = evaluate_array(model, X_perm, y_test, batch_size=256)
    timebin_rows.append({
        "bin_id": b,
        "hours": f"{4*b}-{4*(b+1)}h",
        "AUROC_drop": baseline_metrics["AUROC"] - m["AUROC"],
        "AUPRC_drop": baseline_metrics["AUPRC"] - m["AUPRC"],
        "Brier_increase": m["Brier"] - baseline_metrics["Brier"],
    })

timebin_df = pd.DataFrame(timebin_rows)
timebin_df.to_csv(RESULTS_DIR / "medformer_timebin_importance.csv", index=False)
timebin_df

In [ ]:
# ============================================================
# 14. Medformer feature × time-bin importance
# ============================================================

RUN_FEATURE_TIME_IMPORTANCE = True

if RUN_FEATURE_TIME_IMPORTANCE:
    baseline_auprc = baseline_metrics["AUPRC"]
    rows = []

    for b in range(SEQ_LEN):
        for j, var in enumerate(DYNAMIC_VARS):
            X_perm = X_test_seq.copy()
            X_perm[:, b, j] = 0.0
            X_perm[:, b, j + len(DYNAMIC_VARS)] = 0.0
            m = evaluate_array(model, X_perm, y_test, batch_size=256)
            rows.append({
                "bin_id": b,
                "hours": f"{4*b}-{4*(b+1)}h",
                "Parameter": var,
                "AUPRC_drop": baseline_auprc - m["AUPRC"],
                "AUROC_drop": baseline_metrics["AUROC"] - m["AUROC"],
                "Brier_increase": m["Brier"] - baseline_metrics["Brier"],
            })

    feature_time_importance = pd.DataFrame(rows)
    feature_time_importance.to_csv(RESULTS_DIR / "medformer_feature_time_importance.csv", index=False)
    print("Saved medformer_feature_time_importance.csv")
else:
    print("Skipped feature-time importance.")

In [ ]:
# ============================================================
# 15. Markov chain severity score and state construction
# ============================================================

var_idx = {v: i for i, v in enumerate(DYNAMIC_VARS)}

severity_components = [
    {"component": "low_GCS", "rule": "GCS <= 12 adds 1; GCS <= 8 adds additional 1"},
    {"component": "low_MAP", "rule": "MAP < 65 adds 1"},
    {"component": "low_SysABP", "rule": "SysABP < 90 adds 1"},
    {"component": "abnormal_HR", "rule": "HR > 120 or HR < 50 adds 1"},
    {"component": "abnormal_RespRate", "rule": "RespRate > 24 or RespRate < 8 adds 1"},
    {"component": "high_Creatinine", "rule": "Creatinine > 1.5 adds 1; > 2.0 adds additional 1"},
    {"component": "high_BUN", "rule": "BUN > 25 adds 1"},
    {"component": "low_Urine", "rule": "Urine < 120 within a 4-hour bin adds 1"},
    {"component": "high_Lactate", "rule": "Lactate > 2 adds 1; > 4 adds additional 1"},
    {"component": "abnormal_pH", "rule": "pH < 7.35 or > 7.45 adds 1; pH < 7.25 adds additional 1"},
    {"component": "abnormal_Temp", "rule": "Temp < 36 or > 38.5 adds 1"},
    {"component": "abnormal_K", "rule": "K < 3.0 or > 5.5 adds 1"},
    {"component": "abnormal_Na", "rule": "Na < 130 or > 150 adds 1"},
]
pd.DataFrame(severity_components).to_csv(RESULTS_DIR / "markov_severity_components.csv", index=False)

def get_var(X, name):
    if name not in var_idx:
        return None
    return X[:, :, var_idx[name]]

def add_condition(score, condition, weight=1.0):
    condition = np.where(np.isnan(condition), False, condition)
    score += condition.astype(np.float32) * weight
    return score

def compute_severity_score(X_value):
    score = np.zeros((X_value.shape[0], X_value.shape[1]), dtype=np.float32)

    GCS = get_var(X_value, "GCS")
    MAP = get_var(X_value, "MAP")
    SysABP = get_var(X_value, "SysABP")
    HR = get_var(X_value, "HR")
    RR = get_var(X_value, "RespRate")
    Creatinine = get_var(X_value, "Creatinine")
    BUN = get_var(X_value, "BUN")
    Urine = get_var(X_value, "Urine")
    Lactate = get_var(X_value, "Lactate")
    pH = get_var(X_value, "pH")
    Temp = get_var(X_value, "Temp")
    K = get_var(X_value, "K")
    Na = get_var(X_value, "Na")

    if GCS is not None:
        score = add_condition(score, GCS <= 12, 1)
        score = add_condition(score, GCS <= 8, 1)
    if MAP is not None:
        score = add_condition(score, MAP < 65, 1)
    if SysABP is not None:
        score = add_condition(score, SysABP < 90, 1)
    if HR is not None:
        score = add_condition(score, (HR > 120) | (HR < 50), 1)
    if RR is not None:
        score = add_condition(score, (RR > 24) | (RR < 8), 1)
    if Creatinine is not None:
        score = add_condition(score, Creatinine > 1.5, 1)
        score = add_condition(score, Creatinine > 2.0, 1)
    if BUN is not None:
        score = add_condition(score, BUN > 25, 1)
    if Urine is not None:
        score = add_condition(score, Urine < 120, 1)
    if Lactate is not None:
        score = add_condition(score, Lactate > 2, 1)
        score = add_condition(score, Lactate > 4, 1)
    if pH is not None:
        score = add_condition(score, (pH < 7.35) | (pH > 7.45), 1)
        score = add_condition(score, pH < 7.25, 1)
    if Temp is not None:
        score = add_condition(score, (Temp < 36) | (Temp > 38.5), 1)
    if K is not None:
        score = add_condition(score, (K < 3.0) | (K > 5.5), 1)
    if Na is not None:
        score = add_condition(score, (Na < 130) | (Na > 150), 1)

    return score

score_train = compute_severity_score(X_train_value_raw_split)
score_val = compute_severity_score(X_val_value_raw)
score_test = compute_severity_score(X_test_value_raw)

# Learn state cutoffs from the training split only.
flat_train_scores = score_train.reshape(-1)
cutoffs = np.quantile(flat_train_scores, [0.25, 0.50, 0.75])
cutoffs = np.asarray(cutoffs, dtype=np.float32)

def assign_states(score, cutoffs):
    return np.digitize(score, bins=cutoffs, right=True).astype(int)

states_train = assign_states(score_train, cutoffs)
states_val = assign_states(score_val, cutoffs)
states_test = assign_states(score_test, cutoffs)

pd.DataFrame({
    "cutoff_name": ["q25", "q50", "q75"],
    "severity_score_cutoff": cutoffs,
}).to_csv(RESULTS_DIR / "markov_state_cutoffs.csv", index=False)

print("Markov cutoffs:", cutoffs)
print("Train states shape:", states_train.shape)
print("Test states shape :", states_test.shape)

In [ ]:
# ============================================================
# 16. Save Markov state sequence and severity score long tables
# ============================================================

def save_state_sequence(states, scores, record_ids, y, split_name):
    seq_df = pd.DataFrame({"RecordID": record_ids, "y_true": y, "split": split_name})
    for b in range(states.shape[1]):
        seq_df[f"state_bin_{b}"] = states[:, b]
        seq_df[f"score_bin_{b}"] = scores[:, b]
    seq_df.to_csv(RESULTS_DIR / f"markov_state_sequences_{split_name}.csv", index=False)

    long_rows = []
    for i, rid in enumerate(record_ids):
        for b in range(states.shape[1]):
            long_rows.append({
                "RecordID": rid,
                "y_true": int(y[i]),
                "split": split_name,
                "bin_id": b,
                "hours": f"{4*b}-{4*(b+1)}h",
                "severity_score": float(scores[i, b]),
                "state": int(states[i, b]),
                "state_label": STATE_LABELS[int(states[i, b])],
            })
    long_df = pd.DataFrame(long_rows)
    long_df.to_csv(RESULTS_DIR / f"markov_severity_long_{split_name}.csv", index=False)
    return seq_df, long_df

seq_train_df, long_train_df = save_state_sequence(states_train, score_train, record_ids_train, y_train, "train")
seq_val_df, long_val_df = save_state_sequence(states_val, score_val, record_ids_val, y_val, "val")
seq_test_df, long_test_df = save_state_sequence(states_test, score_test, record_ids_test, y_test, "test_C")

markov_long_all = pd.concat([long_train_df, long_val_df, long_test_df], ignore_index=True)
markov_long_all.to_csv(RESULTS_DIR / "markov_severity_long_all.csv", index=False)

print("Saved Markov state and severity tables.")

In [ ]:
# ============================================================
# 17. Markov transition matrices and state occupancy summaries
# ============================================================

def transition_matrix(states_subset):
    counts = np.zeros((4, 4), dtype=float)
    for seq in states_subset:
        for t in range(seq.shape[0] - 1):
            counts[int(seq[t]), int(seq[t + 1])] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    probs = np.divide(counts, row_sums, out=np.zeros_like(counts), where=row_sums > 0)
    return counts, probs

def transition_table_for_split(states, y, split_name):
    rows = []
    groups = {
        "all": np.ones_like(y, dtype=bool),
        "survivor": y == 0,
        "death": y == 1,
    }
    for group_name, mask in groups.items():
        counts, probs = transition_matrix(states[mask])
        for i in range(4):
            for j in range(4):
                rows.append({
                    "split": split_name,
                    "outcome_group": group_name,
                    "from_state": i,
                    "to_state": j,
                    "from_state_label": STATE_LABELS[i],
                    "to_state_label": STATE_LABELS[j],
                    "count": counts[i, j],
                    "prob": probs[i, j],
                })
    return pd.DataFrame(rows)

transition_df = pd.concat([
    transition_table_for_split(states_train, y_train, "train"),
    transition_table_for_split(states_val, y_val, "val"),
    transition_table_for_split(states_test, y_test, "test_C"),
], ignore_index=True)
transition_df.to_csv(RESULTS_DIR / "markov_transition_matrices.csv", index=False)

def state_distribution_table(states, y, split_name):
    rows = []
    groups = {
        "all": np.ones_like(y, dtype=bool),
        "survivor": y == 0,
        "death": y == 1,
    }
    for group_name, mask in groups.items():
        flat = states[mask].reshape(-1)
        total = len(flat)
        for s in range(4):
            count = int((flat == s).sum())
            rows.append({
                "split": split_name,
                "outcome_group": group_name,
                "state": s,
                "state_label": STATE_LABELS[s],
                "count": count,
                "proportion": count / total if total else np.nan,
            })
    return pd.DataFrame(rows)

state_dist_df = pd.concat([
    state_distribution_table(states_train, y_train, "train"),
    state_distribution_table(states_val, y_val, "val"),
    state_distribution_table(states_test, y_test, "test_C"),
], ignore_index=True)
state_dist_df.to_csv(RESULTS_DIR / "markov_state_distribution.csv", index=False)

def state_occupancy_over_time(states, y, split_name):
    rows = []
    groups = {
        "all": np.ones_like(y, dtype=bool),
        "survivor": y == 0,
        "death": y == 1,
    }
    for group_name, mask in groups.items():
        st = states[mask]
        for b in range(st.shape[1]):
            total = st.shape[0]
            for s in range(4):
                rows.append({
                    "split": split_name,
                    "outcome_group": group_name,
                    "bin_id": b,
                    "hours": f"{4*b}-{4*(b+1)}h",
                    "state": s,
                    "state_label": STATE_LABELS[s],
                    "proportion": float((st[:, b] == s).sum() / total) if total else np.nan,
                })
    return pd.DataFrame(rows)

state_occ_df = pd.concat([
    state_occupancy_over_time(states_train, y_train, "train"),
    state_occupancy_over_time(states_val, y_val, "val"),
    state_occupancy_over_time(states_test, y_test, "test_C"),
], ignore_index=True)
state_occ_df.to_csv(RESULTS_DIR / "markov_state_occupancy.csv", index=False)

print("Saved Markov transition and occupancy summaries.")

In [ ]:
# ============================================================
# 18. Markov trajectory features
# ============================================================

def entropy_of_states(seq):
    counts = np.bincount(seq.astype(int), minlength=4).astype(float)
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def extract_markov_features(states, record_ids, y, split_name):
    rows = []
    for i, seq in enumerate(states):
        seq = seq.astype(int)
        diffs = np.diff(seq)
        row = {
            "RecordID": int(record_ids[i]),
            "y_true": int(y[i]),
            "split": split_name,
            "initial_state": int(seq[0]),
            "final_state": int(seq[-1]),
            "max_state": int(seq.max()),
            "mean_state": float(seq.mean()),
            "delta_final_initial": int(seq[-1] - seq[0]),
            "num_worsening_transitions": int((diffs > 0).sum()),
            "num_recovery_transitions": int((diffs < 0).sum()),
            "worsening_rate": float((diffs > 0).mean()),
            "recovery_rate": float((diffs < 0).mean()),
            "state_entropy": entropy_of_states(seq),
            "ever_critical": int((seq == 3).any()),
            "time_in_critical": float((seq == 3).mean()),
            "early_mean_state": float(seq[:6].mean()),
            "late_mean_state": float(seq[6:].mean()),
            "late_minus_early_mean_state": float(seq[6:].mean() - seq[:6].mean()),
        }
        for s in range(4):
            row[f"time_in_state_{s}"] = float((seq == s).mean())

        counts, probs = transition_matrix(seq.reshape(1, -1))
        for a in range(4):
            for b in range(4):
                row[f"N_{a}_to_{b}"] = float(counts[a, b])
                row[f"P_{a}_to_{b}"] = float(probs[a, b])
        rows.append(row)
    return pd.DataFrame(rows)

markov_features_train = extract_markov_features(states_train, record_ids_train, y_train, "train")
markov_features_val = extract_markov_features(states_val, record_ids_val, y_val, "val")
markov_features_test = extract_markov_features(states_test, record_ids_test, y_test, "test_C")

markov_features_train.to_csv(RESULTS_DIR / "markov_trajectory_features_train.csv", index=False)
markov_features_val.to_csv(RESULTS_DIR / "markov_trajectory_features_val.csv", index=False)
markov_features_test.to_csv(RESULTS_DIR / "markov_trajectory_features_test.csv", index=False)

markov_features_all = pd.concat([markov_features_train, markov_features_val, markov_features_test], ignore_index=True)
markov_features_all.to_csv(RESULTS_DIR / "markov_trajectory_features_all.csv", index=False)

print("Markov feature shape:", markov_features_train.shape, markov_features_test.shape)

In [ ]:
# ============================================================
# 19. Markov-only mortality prediction models
# ============================================================

markov_feature_cols = [
    c for c in markov_features_train.columns
    if c not in ["RecordID", "y_true", "split"]
]

X_m_train = markov_features_train[markov_feature_cols].fillna(0.0).to_numpy()
y_m_train = markov_features_train["y_true"].to_numpy()
X_m_val = markov_features_val[markov_feature_cols].fillna(0.0).to_numpy()
y_m_val = markov_features_val["y_true"].to_numpy()
X_m_test = markov_features_test[markov_feature_cols].fillna(0.0).to_numpy()
y_m_test = markov_features_test["y_true"].to_numpy()

scaler_m = StandardScaler()
X_m_train_s = scaler_m.fit_transform(X_m_train)
X_m_val_s = scaler_m.transform(X_m_val)
X_m_test_s = scaler_m.transform(X_m_test)

markov_models = {
    "Markov_LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    "Markov_RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
}

markov_metric_rows = []
markov_pred_rows = []

for model_name, clf in markov_models.items():
    clf.fit(X_m_train_s, y_m_train)
    p_test_m = clf.predict_proba(X_m_test_s)[:, 1]
    y_pred_m = (p_test_m >= 0.5).astype(int)

    metrics = {
        "model": model_name,
        "AUROC": roc_auc_score(y_m_test, p_test_m),
        "AUPRC": average_precision_score(y_m_test, p_test_m),
        "Accuracy": accuracy_score(y_m_test, y_pred_m),
        "Precision": precision_score(y_m_test, y_pred_m, zero_division=0),
        "Recall": recall_score(y_m_test, y_pred_m, zero_division=0),
        "F1": f1_score(y_m_test, y_pred_m, zero_division=0),
        "Brier": brier_score_loss(y_m_test, p_test_m),
    }
    markov_metric_rows.append(metrics)

    for rid, yt, pp, yp in zip(markov_features_test["RecordID"], y_m_test, p_test_m, y_pred_m):
        markov_pred_rows.append({
            "RecordID": rid,
            "y_true": yt,
            "p_mortality": pp,
            "y_pred": yp,
            "model": model_name,
            "split": "test_C",
        })

markov_model_metrics = pd.DataFrame(markov_metric_rows)
markov_model_predictions = pd.DataFrame(markov_pred_rows)

markov_model_metrics.to_csv(RESULTS_DIR / "markov_model_metrics.csv", index=False)
markov_model_predictions.to_csv(RESULTS_DIR / "markov_model_predictions_test.csv", index=False)

markov_model_metrics

In [ ]:
# ============================================================
# 20. Markov feature standardized mean differences
# ============================================================

def compute_smd_table(df, feature_cols, split_name):
    rows = []
    d0 = df[df["y_true"] == 0]
    d1 = df[df["y_true"] == 1]
    for col in feature_cols:
        x0 = d0[col].astype(float).to_numpy()
        x1 = d1[col].astype(float).to_numpy()
        m0, m1 = np.nanmean(x0), np.nanmean(x1)
        s0, s1 = np.nanstd(x0), np.nanstd(x1)
        pooled = math.sqrt((s0**2 + s1**2) / 2)
        smd = (m1 - m0) / pooled if pooled > 1e-12 else 0.0
        rows.append({
            "split": split_name,
            "feature": col,
            "mean_survivor": m0,
            "mean_death": m1,
            "smd_death_minus_survivor": smd,
            "abs_smd": abs(smd),
        })
    return pd.DataFrame(rows).sort_values("abs_smd", ascending=False)

smd_train = compute_smd_table(markov_features_train, markov_feature_cols, "train")
smd_test = compute_smd_table(markov_features_test, markov_feature_cols, "test_C")
smd_df = pd.concat([smd_train, smd_test], ignore_index=True)
smd_df.to_csv(RESULTS_DIR / "markov_feature_smd.csv", index=False)

smd_df.head(20)

In [ ]:
# ============================================================
# 21. Integrated Medformer–Markov patient-level table
# ============================================================

integrated_test = medformer_pred_selected.merge(
    markov_features_test,
    on=["RecordID", "y_true"],
    how="left",
    suffixes=("", "_markov"),
)

# Risk quartiles within external test for interpretation.
integrated_test["medformer_risk_quartile"] = pd.qcut(
    integrated_test["p_mortality"],
    q=4,
    labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"],
    duplicates="drop",
)
integrated_test["medformer_risk_quartile"] = integrated_test["medformer_risk_quartile"].astype(str)

integrated_test.to_csv(RESULTS_DIR / "patient_temporal_summary_test_c.csv", index=False)

# Risk-quartile Markov state occupancy
seq_test_for_risk = seq_test_df.merge(
    integrated_test[["RecordID", "medformer_risk_quartile", "p_mortality"]],
    on="RecordID",
    how="left",
)

risk_occ_rows = []
for quartile, qdf in seq_test_for_risk.groupby("medformer_risk_quartile"):
    if quartile == "nan":
        continue
    for b in range(SEQ_LEN):
        values = qdf[f"state_bin_{b}"].astype(int).to_numpy()
        for s in range(4):
            risk_occ_rows.append({
                "risk_quartile": quartile,
                "bin_id": b,
                "hours": f"{4*b}-{4*(b+1)}h",
                "state": s,
                "state_label": STATE_LABELS[s],
                "proportion": float((values == s).mean()),
            })
risk_occ_df = pd.DataFrame(risk_occ_rows)
risk_occ_df.to_csv(RESULTS_DIR / "integrated_state_occupancy_by_risk_quartile.csv", index=False)

# Correlation between Medformer risk and Markov features
corr_rows = []
for col in markov_feature_cols:
    x = integrated_test[col].astype(float).to_numpy()
    y = integrated_test["p_mortality"].astype(float).to_numpy()
    if np.nanstd(x) < 1e-12:
        corr = 0.0
    else:
        corr = np.corrcoef(x, y)[0, 1]
    corr_rows.append({
        "feature": col,
        "correlation_with_medformer_risk": float(corr),
        "abs_correlation": float(abs(corr)),
    })
corr_df = pd.DataFrame(corr_rows).sort_values("abs_correlation", ascending=False)
corr_df.to_csv(RESULTS_DIR / "medformer_markov_correlations.csv", index=False)

# Transition matrices by Medformer low/high risk quartile
low_mask = seq_test_for_risk["medformer_risk_quartile"] == "Q1_lowest"
high_mask = seq_test_for_risk["medformer_risk_quartile"] == "Q4_highest"

risk_transition_rows = []
for group_name, mask in {"Q1_lowest": low_mask, "Q4_highest": high_mask}.items():
    state_cols = [f"state_bin_{b}" for b in range(SEQ_LEN)]
    states_group = seq_test_for_risk.loc[mask, state_cols].to_numpy(dtype=int)
    counts, probs = transition_matrix(states_group)
    for i in range(4):
        for j in range(4):
            risk_transition_rows.append({
                "risk_group": group_name,
                "from_state": i,
                "to_state": j,
                "from_state_label": STATE_LABELS[i],
                "to_state_label": STATE_LABELS[j],
                "count": counts[i, j],
                "prob": probs[i, j],
            })

risk_transition_df = pd.DataFrame(risk_transition_rows)
risk_transition_df.to_csv(RESULTS_DIR / "markov_transition_by_medformer_risk.csv", index=False)

print("Integrated test table:", integrated_test.shape)
print("Saved integrated outputs.")

In [ ]:
# ============================================================
# 22. Final checklist
# ============================================================

expected_files = [
    "medformer_test_predictions.csv",
    "medformer_test_metrics.csv",
    "medformer_training_history.csv",
    "medformer_timebin_importance.csv",
    "medformer_feature_time_importance.csv",
    "markov_state_sequences_test_C.csv",
    "markov_transition_matrices.csv",
    "markov_trajectory_features_test.csv",
    "markov_model_metrics.csv",
    "patient_temporal_summary_test_c.csv",
    "integrated_state_occupancy_by_risk_quartile.csv",
    "medformer_markov_correlations.csv",
    "markov_transition_by_medformer_risk.csv",
]

print("Result folder:", RESULTS_DIR)
for fname in expected_files:
    path = RESULTS_DIR / fname
    print(("OK   " if path.exists() else "MISS "), fname)